## 1. Install Dependencies


In [ ]:
!pip install -q --upgrade datasets pyarrow==14.0.1 pandas
!pip install -q transformers evaluate sacrebleu accelerate wandb
!pip install -q nltk  # For METEOR
!pip install -q unbabel-comet  # For COMET

import os, gc, torch, wandb
import pandas as pd
from datasets import Dataset, concatenate_datasets
from transformers import (
    MBart50TokenizerFast, MBartForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
import evaluate
import torch.nn.functional as F
import shutil
from pathlib import Path

# Download NLTK data for METEOR
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Import COMET
from comet import download_model, load_from_checkpoint

## 2. Setup W&B

In [ ]:
os.environ["WANDB_API_KEY"] = "your-api-key"
wandb.login(key=os.environ["WANDB_API_KEY"])

gc.collect()
torch.cuda.empty_cache()

## 3. Load Training Data

In [ ]:
DATA_DIR = "/kaggle/input/new-dataset"

def load_parallel_csv(file_path, src_col="relevant_sentences", tgt_col="translation_tmg"):
    """
    Load parallel corpus with updated column names
    src_col: Nepali sentences (relevant_sentences)
    tgt_col: Tamang translations (translation_tmg)
    """
    df = pd.read_csv(file_path).dropna(subset=[src_col, tgt_col])
    dataset = Dataset.from_pandas(
        df[[src_col, tgt_col]].rename(columns={src_col: "ne_NP", tgt_col: "tmg"}),
        preserve_index=False
    )
    return dataset

# Load training and testing data
train_ds = load_parallel_csv(os.path.join(DATA_DIR, "train-data.csv"))
test_ds  = load_parallel_csv(os.path.join(DATA_DIR, "test-data.csv"))

print("Training Size:", len(train_ds))
print("Test Size:", len(test_ds))
print("\nSample from training data:")
print(train_ds[0])

## 4. Load Tokenizer & Model

In [ ]:
model_name = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


## 5. Preprocessing

In [ ]:
def preprocess_forward(batch):
    """Nepali → Tamang"""
    tokenizer.src_lang = "ne_NP"
    tokenizer.tgt_lang = "hi_IN"  # Using hi_IN as proxy for Tamang
    model_inputs = tokenizer(batch["ne_NP"], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(text_target=batch["tmg"], max_length=128, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_reverse(batch):
    """Tamang → Nepali"""
    tokenizer.src_lang = "hi_IN"  # Using hi_IN as proxy for Tamang
    tokenizer.tgt_lang = "ne_NP"
    model_inputs = tokenizer(batch["tmg"], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(text_target=batch["ne_NP"], max_length=128, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Process training data (bidirectional)
print("\nPreprocessing training data...")
train_forward = train_ds.map(preprocess_forward, batched=True, remove_columns=["ne_NP", "tmg"])
train_reverse = train_ds.map(preprocess_reverse, batched=True, remove_columns=["ne_NP", "tmg"])

# Process test data (bidirectional)
print("Preprocessing test data...")
test_forward  = test_ds.map(preprocess_forward, batched=True, remove_columns=["ne_NP", "tmg"])
test_reverse  = test_ds.map(preprocess_reverse, batched=True, remove_columns=["ne_NP", "tmg"])

# Combine bidirectional data
train_dataset = concatenate_datasets([train_forward, train_reverse]).shuffle(seed=42)
eval_dataset  = concatenate_datasets([test_forward, test_reverse]).shuffle(seed=42)

print(f"\nFinal training dataset size: {len(train_dataset)} (bidirectional)")
print(f"Final eval dataset size: {len(eval_dataset)} (bidirectional)")

## 6. Setting up all the evaluation metrics (BLEU,chrF,chrF++,METEOR)

In [ ]:
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
meteor = evaluate.load("meteor")

# Load COMET model
print("\nLoading COMET model...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)
print("✓ COMET model loaded successfully!")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU
    result_bleu = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])

    # chrF
    result_chrf = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)

    # chrF++
    result_chrfpp = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)

    # METEOR
    result_meteor = meteor.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": result_bleu["score"],
        "chrf": result_chrf["score"],
        "chrf++": result_chrfpp["score"],
        "meteor": result_meteor["meteor"] * 100  # Convert to percentage
    }

def compute_metrics_with_comet(eval_preds, source_texts):
    """
    Compute all metrics including COMET (requires source texts)
    """
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU
    result_bleu = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])

    # chrF
    result_chrf = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)

    # chrF++
    result_chrfpp = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)

    # METEOR
    result_meteor = meteor.compute(predictions=decoded_preds, references=decoded_labels)

    # COMET (requires source, reference, and hypothesis)
    comet_input = [
        {"src": src, "mt": mt, "ref": ref}
        for src, mt, ref in zip(source_texts, decoded_preds, decoded_labels)
    ]
    comet_output = comet_model.predict(comet_input, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)

    return {
        "bleu": result_bleu["score"],
        "chrf": result_chrf["score"],
        "chrf++": result_chrfpp["score"],
        "meteor": result_meteor["meteor"] * 100,  # Convert to percentage
        "comet": comet_output.system_score * 100  # Convert to percentage
    }

## 7. Training

In [ ]:
output_dir = "./mbartTrans_new_dataset"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=100,
    weight_decay=0.01,
    learning_rate=7e-5,
    fp16=torch.cuda.is_available(),
    optim="adafactor",
    predict_with_generate=True,
    load_best_model_at_end=False,
    metric_for_best_model="bleu",
    report_to="wandb",
    run_name="mbart50_nep_tmg_15k_5ep-7e-5"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

print("\n=== Starting Training ===")
trainer.train()
print("✓ Training completed.")

##  8. Evaluate on Test Data

In [ ]:
print("\n=== Evaluating on Test Data ===")

# Forward direction: Nepali → Tamang
print("\nEvaluating Nepali → Tamang...")
preds_forward = trainer.predict(test_forward)
source_texts_forward = test_ds["ne_NP"]  # Nepali source texts
metrics_forward = compute_metrics_with_comet(
    (preds_forward.predictions, preds_forward.label_ids),
    source_texts_forward
)

# Reverse direction: Tamang → Nepali
print("Evaluating Tamang → Nepali...")
preds_reverse = trainer.predict(test_reverse)
source_texts_reverse = test_ds["tmg"]  # Tamang source texts
metrics_reverse = compute_metrics_with_comet(
    (preds_reverse.predictions, preds_reverse.label_ids),
    source_texts_reverse
)

print("\n" + "="*60)
print("FINAL EVALUATION METRICS ON TEST DATA")
print("="*60)

print("\n📊 Nepali → Tamang:")
print(f"  BLEU:    {metrics_forward['bleu']:.2f}")
print(f"  chrF:    {metrics_forward['chrf']:.2f}")
print(f"  chrF++:  {metrics_forward['chrf++']:.2f}")
print(f"  METEOR:  {metrics_forward['meteor']:.2f}")
print(f"  COMET:   {metrics_forward['comet']:.2f}")

print("\n📊 Tamang → Nepali:")
print(f"  BLEU:    {metrics_reverse['bleu']:.2f}")
print(f"  chrF:    {metrics_reverse['chrf']:.2f}")
print(f"  chrF++:  {metrics_reverse['chrf++']:.2f}")
print(f"  METEOR:  {metrics_reverse['meteor']:.2f}")
print(f"  COMET:   {metrics_reverse['comet']:.2f}")

# Save metrics to file
metrics_dict = {
    "nepali_to_tamang": metrics_forward,
    "tamang_to_nepali": metrics_reverse,
    "dataset_info": {
        "train_size": len(train_ds),
        "test_size": len(test_ds),
        "epochs": 5,
        "learning_rate": 7e-5
    }
}
import json
with open(os.path.join(output_dir, "final_metrics.json"), "w") as f:
    json.dump(metrics_dict, f, indent=2)
print(f"\n✓ Metrics saved to: {output_dir}/final_metrics.json")

## 9. Save Checkpoint

In [ ]:

print("\n=== Preparing Latest Checkpoint for Download ===")

# Find the latest checkpoint
checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
if checkpoint_dirs:
    # Sort by checkpoint number to get the latest
    latest_checkpoint = sorted(checkpoint_dirs, key=lambda x: int(x.split("-")[1]))[-1]
    latest_checkpoint_path = os.path.join(output_dir, latest_checkpoint)

    print(f"✓ Latest checkpoint found: {latest_checkpoint}")

    # Create a directory for the latest checkpoint
    latest_model_dir = "/kaggle/working/latest_checkpoint"
    if os.path.exists(latest_model_dir):
        shutil.rmtree(latest_model_dir)

    # Copy the latest checkpoint
    shutil.copytree(latest_checkpoint_path, latest_model_dir)

    # Also copy the metrics file
    shutil.copy(
        os.path.join(output_dir, "final_metrics.json"),
        os.path.join(latest_model_dir, "final_metrics.json")
    )

    print(f"✓ Latest checkpoint copied to: {latest_model_dir}")

    # Create zip file
    zip_filename = "/kaggle/working/mbart_latest_checkpoint"
    if os.path.exists(f"{zip_filename}.zip"):
        os.remove(f"{zip_filename}.zip")

    shutil.make_archive(zip_filename, 'zip', latest_model_dir)

    print(f"✓ Latest checkpoint zipped successfully: {zip_filename}.zip")
    print(f"✓ File size: {os.path.getsize(f'{zip_filename}.zip') / (1024**3):.2f} GB")

    # Clean up intermediate checkpoints to save space
    print("\n=== Cleaning up intermediate checkpoints ===")
    for checkpoint_dir in checkpoint_dirs[:-1]:  # Keep only the latest
        checkpoint_path = os.path.join(output_dir, checkpoint_dir)
        shutil.rmtree(checkpoint_path)
        print(f"✓ Removed: {checkpoint_dir}")

else:
    print("⚠ No checkpoints found. Saving the current model state instead.")
    latest_model_dir = "/kaggle/working/final_model"
    trainer.save_model(latest_model_dir)
    tokenizer.save_pretrained(latest_model_dir)

    # Copy metrics
    shutil.copy(
        os.path.join(output_dir, "final_metrics.json"),
        os.path.join(latest_model_dir, "final_metrics.json")
    )

    # Create zip
    zip_filename = "/kaggle/working/mbart_final_model"
    shutil.make_archive(zip_filename, 'zip', latest_model_dir)
    print(f"✓ Final model zipped: {zip_filename}.zip")

print("\n" + "="*60)
print("ALL DONE! 🎉")
print("="*60)
print("\nFiles ready for download:")
print(f"  1. {zip_filename}.zip (latest checkpoint)")
print(f"  2. {output_dir}/final_metrics.json (evaluation results)")
print("\nFor Kaggle: Check the output section for the zip file.")
print("\nNote: Only the latest checkpoint is saved to reduce file size.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but y

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Training Size: 15001
Test Size: 5000

Sample from training data:
{'ne_NP': 'सबै विश्वासमा चलेको छ।', 'tmg': 'मोकोन विश्वासरि बोल्बा मुला।'}


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

✓ Training completed.

=== Evaluating on Test Data ===

Evaluating Nepali → Tamang...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|██████████| 625/625 [01:10<00:00,  8.90it/s]


Evaluating Tamang → Nepali...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|██████████| 625/625 [01:04<00:00,  9.69it/s]



FINAL EVALUATION METRICS ON TEST DATA

📊 Nepali → Tamang:
  BLEU:    40.14
  chrF:    72.91
  chrF++:  68.92
  METEOR:  60.46
  COMET:   75.60

📊 Tamang → Nepali:
  BLEU:    42.96
  chrF:    71.65
  chrF++:  68.25
  METEOR:  60.43
  COMET:   79.04

✓ Metrics saved to: ./mbartTrans_new_dataset/final_metrics.json

=== Preparing Latest Checkpoint for Download ===
✓ Latest checkpoint found: checkpoint-18755
✓ Latest checkpoint copied to: /kaggle/working/latest_checkpoint
✓ Latest checkpoint zipped successfully: /kaggle/working/mbart_latest_checkpoint.zip
✓ File size: 2.12 GB

=== Cleaning up intermediate checkpoints ===
✓ Removed: checkpoint-15004
✓ Removed: checkpoint-7502
✓ Removed: checkpoint-3751
✓ Removed: checkpoint-11253

ALL DONE! 🎉

Files ready for download:
  1. /kaggle/working/mbart_latest_checkpoint.zip (latest checkpoint)
  2. ./mbartTrans_new_dataset/final_metrics.json (evaluation results)

For Kaggle: Check the output section for the zip file.

Note: Only the latest checkpo